In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd '/content/drive/MyDrive/Github/Deceptive-Review-Detection'

In [ ]:
!nvidia-smi -L

In [ ]:
pip install transformers

In [ ]:
pip install torchmetrics

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

import torch
import torch.nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from transformers import BertTokenizer, BertModel, BertConfig, AdamW, get_linear_schedule_with_warmup
import sys

In [ ]:
import data_prep
from BERT_Classification import *

In [ ]:
# df = data_prep.import_chi_review()
df = pd.read_csv("Chicago_Hotel_Review/Chicago_Hotel_Reviews.csv", index_col=0)
df = df.drop('Rating', axis=1)
df.head()

In [ ]:
X = df['Ori_Review']
y = df['Label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=0)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=0)
X_train = X_train.to_list()
X_test = X_test.to_list()
X_val = X_val.to_list()
y_train = y_train.to_list()
y_test = y_test.to_list()
y_val = y_val.to_list()

In [ ]:
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print("Device used: {}.".format(device))

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert = BertModel.from_pretrained('bert-base-uncased')

In [ ]:
seq_length = 300
batch_size = 16

In [ ]:
train_dataset = dataset(tokenizer, X_train, y_train, max_seq_length=seq_length)
val_dataset = dataset(tokenizer, X_val, y_val, max_seq_length=seq_length)
test_dataset = dataset(tokenizer, X_test, y_test, max_seq_length=seq_length)

In [ ]:
train_data_loader = DataLoader(dataset=train_dataset, 
                         batch_size=batch_size, 
                         shuffle=True)
val_data_loader = DataLoader(dataset=val_dataset, 
                         batch_size=batch_size, 
                         shuffle=True)
test_data_loader = DataLoader(dataset=test_dataset, 
                         batch_size=batch_size, 
                         shuffle=True)

In [ ]:
model = BertWithClassifier(bert, seq_length, nn.CrossEntropyLoss).to(device)

In [ ]:
lrs = [5e-5, 3e-5, 2e-5]
batch_sizes = [16, 32]
num_epochs = [2, 3, 4]

In [ ]:
train_losses, val_losses = train(model=model, 
                                data_loader=train_data_loader, 
                                val_data_loader=val_data_loader, 
                                optimizer=AdamW, 
                                learning_rate=3e-5,
                                num_epochs=30,
                                device=device,
                                scheduler=get_linear_schedule_with_warmup,
                                batch_size=batch_size,
                                data_size=len(train_dataset))

In [ ]:
train_losses, val_losses

In [ ]:
# test_preds, test_loss, test_acc = validate(model, test_data_loader, device)

In [ ]:
# test_acc